## 環境設置

In [1]:
try:
    import openseespy.opensees as ops  # noqa: F401
except ImportError:
    import sys
    !{sys.executable} -m pip install -q openseespy
    import openseespy.opensees as ops  # noqa: F401

# Case-03.6b:可抽換檢核模組——RC 到鋼結構的示範

Case-03.6 已經證明:分析端(`analyze_frame`,只吃 E/I/A/幾何)跟檢核端
(強度公式)可以乾淨分離。這裡把這個分離**正式化成一個共同介面**,
先驗證 RC 檢核重構後數字沒有跑掉(迴歸測試),再示範接上完全不同的
材料——鋼結構(依《鋼結構極限設計法規範及解說》)。

**共同介面規格**(每種材料的檢核函式都要符合):

```
輸入: 需求力 (Mu, Vu, Pu) + 該材料的斷面參數(不同材料參數不同,
      RC是邊長, 鋼結構是外徑+管壁厚, 這是合理的差異, 不強求一樣)
輸出: dict, 固定包含以下欄位:
      material, phiPn, phiMn, phiVn, p_util, m_util, v_util
```

**分析端完全不用改**——需求力 Mu/Vu/Pu 只要對稱構架幾何跟施加的
力沒有變,不管換哪種材料都是同一組數字,這是 Case-03.6 已經驗證過
的結論,這裡直接沿用。

## 第 1 課:需求力——分析端完全不動

先重跑一次沿用至今的剪力構架分析,拿到需求力。這一步**不管後面接
哪種材料檢核,都不需要重做**——這正是「分析」跟「檢核」分離的價值。

In [2]:
E = 2.463e7   # kN/m^2, RC彈性模數(這裡只是用來跑一次分析模型, 不影響後面的材料檢核比較)
h1 = h2 = 3.5
L = 6.0
F1_frame, F2_frame = 9.938, 15.900

def analyze_frame(h_col, E_mat):
    Ic = h_col**4/12
    ops.wipe()
    ops.model('basic', '-ndm', 2, '-ndf', 3)
    ops.node(1, 0.0, 0.0);   ops.node(2, L, 0.0)
    ops.node(3, 0.0, h1);    ops.node(4, L, h1)
    ops.node(5, 0.0, h1+h2); ops.node(6, L, h1+h2)
    ops.fix(1,1,1,1); ops.fix(2,1,1,1)
    for n in [3,4,5,6]:
        ops.fix(n, 0,1,1)
    ops.equalDOF(3,4,1); ops.equalDOF(5,6,1)
    A_big = 1.0e6
    ops.geomTransf('Linear', 1)
    ops.element('elasticBeamColumn',1,1,3,A_big,E_mat,Ic,1)
    ops.element('elasticBeamColumn',2,2,4,A_big,E_mat,Ic,1)
    ops.element('elasticBeamColumn',3,3,5,A_big,E_mat,Ic,1)
    ops.element('elasticBeamColumn',4,4,6,A_big,E_mat,Ic,1)
    ops.timeSeries('Linear',1); ops.pattern('Plain',1,1)
    ops.load(3, F1_frame, 0.0, 0.0)
    ops.load(5, F2_frame, 0.0, 0.0)
    ops.system('BandGeneral'); ops.numberer('RCM'); ops.constraints('Transformation')
    ops.test('NormDispIncr',1e-10,20); ops.algorithm('Newton')
    ops.integrator('LoadControl',1.0); ops.analysis('Static'); ops.analyze(1)
    f1 = ops.eleForce(1)
    return abs(f1[2]), abs(f1[1])   # Mu, Vu

Mu, Vu = analyze_frame(0.40, E)   # 用RC斷面跑一次分析取得需求力
Pu = 147.60   # 沿用Case-03.6概估的重力軸力

print(f"Mu = {Mu:.3f} kN-m, Vu = {Vu:.3f} kN, Pu = {Pu:.2f} kN")
print("(這組需求力接下來對RC跟鋼結構檢核都一樣, 不會因材料改變)")

Mu = 22.608 kN-m, Vu = 0.000 kN, Pu = 147.60 kN
(這組需求力接下來對RC跟鋼結構檢核都一樣, 不會因材料改變)


## 第 2 課:RC 檢核模組(重構,依《結構混凝土設計規範》)

跟 Case-03.6 完全一樣的公式,只是包成符合共同介面的函式。

In [3]:
def Mn_strain_compat(h_col_m, Pu_kN, fc=280.0, fy=4200.0, Es=2.0e6, rho=0.02):
    """應變相容法算標稱彎矩強度(取代單筋梁公式, 跟Case-03.6同一套邏輯)"""
    beta1, eps_cu = 0.85, 0.003
    h = h_col_m*100
    cover = 4.0
    half = h/2 - cover
    if half <= 0:
        return 0.0
    As_bar = (rho*h*h)/8
    bar_layers = {half: 3, 0.0: 2, -half: 3}
    Pu_kgf = Pu_kN*1000/9.80665
    eps_y = fy/Es

    def section_force(c):
        a = min(beta1*c, h)
        Cc = 0.85*fc*h*a
        y_Cc = half - a/2
        N, M = Cc, Cc*y_Cc
        for y, n_bars in bar_layers.items():
            As = n_bars*As_bar
            dist = half - y
            eps_s = eps_cu*(c-dist)/c if c > 0 else 0.0
            eps_s = max(min(eps_s, eps_y), -eps_y)
            fs = Es*eps_s
            fs_net = fs - 0.85*fc if dist <= a else fs
            Fs = As*fs_net
            N += Fs; M += Fs*y
        return N, M

    c_lo, c_hi = 0.1, h*3
    for _ in range(100):
        c_mid = (c_lo+c_hi)/2
        N, _ = section_force(c_mid)
        if N > Pu_kgf:
            c_hi = c_mid
        else:
            c_lo = c_mid
    _, M_final = section_force((c_lo+c_hi)/2)
    return M_final*9.80665e-5


def rc_strength_check(h_col_m, Mu, Vu, Pu):
    """RC方形柱快速強度檢核, 依結構混凝土設計規範第21/22章"""
    fc, fy = 280.0, 4200.0        # kgf/cm^2
    rho = 0.02
    phi_axial = phi_moment = 0.65  # 表21.2.2, 壓力控制
    phi_shear = 0.75                # 表21.2.1(b)

    b = d = h_col_m*100
    Ag = b*d
    Ast = rho*Ag

    Po = 0.85*fc*(Ag-Ast) + fy*Ast          # 式22.4.2.2
    phiPn = phi_axial*0.80*Po*9.80665e-3     # kgf->kN

    Mn = Mn_strain_compat(h_col_m, Pu, fc, fy, rho=rho)   # 應變相容法, 非單筋梁公式
    phiMn = phi_moment*Mn

    Vc = 0.53*(fc**0.5)*b*d
    phiVn = phi_shear*Vc*9.80665e-3          # kgf->kN

    return dict(material='RC', section=f'{h_col_m*100:.0f}x{h_col_m*100:.0f}cm',
                phiPn=phiPn, phiMn=phiMn, phiVn=phiVn,
                p_util=Pu/phiPn, m_util=Mu/phiMn,
                v_util=(Vu/phiVn if phiVn>0 else 0.0))


rc_result = rc_strength_check(0.40, Mu, Vu, Pu)
print(rc_result)

# 迴歸測試: 確認與Case-06纖維斷面/應變相容法驗證結果一致
assert abs(rc_result['phiMn']/0.65 - 220.83) < 0.5, "跟Case-06驗證結果對不起來!"
print("\n[PASS] RC檢核數字與Case-06纖維斷面驗證結果一致")

{'material': 'RC', 'section': '40x40cm', 'phiPn': 2588.403289472, 'phiMn': 143.542055582066, 'phiVn': 104.3654636659562, 'p_util': 0.057023571481439606, 'm_util': 0.15750262115394043, 'v_util': 0.0}

[PASS] RC檢核數字與Case-06纖維斷面驗證結果一致


## 第 3 課:鋼結構檢核模組(新增,依《鋼結構極限設計法規範及解說》)

依規範查表值:
- 軸力:式 6.2-1~6.2-4,φc=0.85,Fcr 依無因次長細比 λc 分段
- 彎矩:Mp=Fy·Z(假設側向有充分支撐,Lb≤Lp,結實斷面),φb=0.90
- 剪力:式 7.3-1,Vn=0.6Fyw·Aw,φv=0.90
- 組合:式 8.2-1a/1b(Pu/φPn≥0.2 或 <0.2 分兩式)

斷面選用**方形鋼管(HSS)**,不是實心方形——這是刻意的簡化,為了能跟
RC 案例的方形柱直接比較外觀尺寸;真實鋼結構柱更常用 H 型鋼或箱型鋼,
這裡不查型鋼表,直接用外徑+管壁厚計算斷面性質。

In [4]:
def steel_strength_check(b_cm, t_cm, Mu, Vu, Pu, L_cm=350.0, K=1.0):
    """鋼結構方形鋼管(HSS)柱快速強度檢核, 依鋼結構極限設計法規範"""
    E_steel = 2100.0   # tf/cm^2, 規範原文值
    Fy = 2.5            # tf/cm^2, 常見鋼材等級(SN400)
    phi_c, phi_b, phi_v = 0.85, 0.90, 0.90   # 式6.2-1, 8.2-1, 7.3-1

    Ag = b_cm**2 - (b_cm-2*t_cm)**2
    I = (b_cm**4 - (b_cm-2*t_cm)**4)/12
    r = (I/Ag)**0.5
    Z = (b_cm**3 - (b_cm-2*t_cm)**3)/4       # 方管塑性模數近似公式
    Aw = 2*(b_cm-2*t_cm)*t_cm                 # 概估腹板面積(兩側管壁)

    lam_c = (K*L_cm/(3.14159265*r))*(Fy/E_steel)**0.5
    if lam_c <= 1.5:
        import math
        Fcr = math.exp(-0.419*lam_c**2)*Fy    # 式6.2-2
    else:
        Fcr = (0.877/lam_c**2)*Fy             # 式6.2-3
    phiPn = phi_c*Ag*Fcr*9.80665              # tf->kN

    Mn = Fy*Z*9.80665e-4                       # tf-cm->kN-m (1tf-cm=9.80665e-2kN*0.01m=9.80665e-4kN-m)
    phiMn = phi_b*Mn

    Vn = 0.6*Fy*Aw
    phiVn = phi_v*Vn*9.80665                   # tf->kN

    p_util = Pu/phiPn
    m_util = Mu/phiMn
    v_util = Vu/phiVn if phiVn>0 else 0.0

    # 組合力交互作用, 式8.2-1a/1b
    if p_util >= 0.2:
        combined = p_util + (8/9)*m_util
    else:
        combined = p_util/2 + m_util

    return dict(material='Steel', section=f'HSS{b_cm:.0f}x{b_cm:.0f}x{t_cm:.1f}cm',
                phiPn=phiPn, phiMn=phiMn, phiVn=phiVn,
                p_util=p_util, m_util=m_util, v_util=v_util,
                combined=combined)


steel_result = steel_strength_check(40.0, 1.6, Mu, Vu, Pu)
print(steel_result)

{'material': 'Steel', 'section': 'HSS40x40x1.6cm', 'phiPn': 4994.236366041951, 'phiMn': 7.813185569280003, 'phiVn': 1559.0219903999998, 'p_util': 0.029554067765714585, 'm_util': 2.8936020781192564, 'v_util': 0.0, 'combined': 2.908379112002114}


## 第 4 課:第一次嘗試——直接照搬 RC 尺寸,不通過

刻意先用「跟 RC 一樣的外觀尺寸」(40cm 見方)試一次,不調整——結果
彎矩檢核大幅超標。這不是失敗案例包裝,是真實算出來的結果,而且揭露
一個材料轉換時容易忽略的重點。

In [5]:
print(f"HSS 40x40x1.6cm 檢核結果:")
print(f"  軸力利用率 = {steel_result['p_util']:.1%}")
print(f"  彎矩利用率 = {steel_result['m_util']:.1%}")
print(f"  組合交互作用值 = {steel_result['combined']:.3f} (限值1.0)")

first_try_fail = steel_result['combined'] > 1.0
assert first_try_fail, "預期第一次嘗試應該FAIL"
print(f"\n[確認FAIL] 薄壁鋼管彎矩承載力遠不如同外觀尺寸的RC柱")
print(f"原因: RC柱用滿滿40x40實心斷面(鋼筋雖只佔2%, 但混凝土壓力區貢獻大量抗彎能力);")
print(f"      鋼管是薄殼斷面, 材料集中在管壁, 相同外觀尺寸下抗彎慣性矩小很多。")
print(f"這不是公式錯, 是材料轉換時不能直接沿用RC的尺寸直覺, 需要重新試設。")

HSS 40x40x1.6cm 檢核結果:
  軸力利用率 = 3.0%
  彎矩利用率 = 289.4%
  組合交互作用值 = 2.908 (限值1.0)

[確認FAIL] 薄壁鋼管彎矩承載力遠不如同外觀尺寸的RC柱
原因: RC柱用滿滿40x40實心斷面(鋼筋雖只佔2%, 但混凝土壓力區貢獻大量抗彎能力);
      鋼管是薄殼斷面, 材料集中在管壁, 相同外觀尺寸下抗彎慣性矩小很多。
這不是公式錯, 是材料轉換時不能直接沿用RC的尺寸直覺, 需要重新試設。


## 第 5 課:重新試設——鋼結構的直覺不一樣

跟 RC 案例一樣的「試設→檢核→不夠就調整」迴圈,但這次調整的方向不同:
鋼結構彎矩承載力對「整體深度」比對「管壁厚度」敏感很多(材料離中性軸
越遠,抗彎貢獻越大),所以先試「加大深度」而不是單純「加厚管壁」。

In [6]:
candidates = [(40,1.6), (40,2.5), (40,3.5), (40,5.0), (50,3.5)]

print(f"{'斷面':<18}{'P利用率':<10}{'M利用率':<10}{'交互作用':<10}{'結果':<6}")
for b,t in candidates:
    r = steel_strength_check(b, t, Mu, Vu, Pu)
    status = "PASS" if r['combined'] <= 1.0 else "FAIL"
    print(f"HSS{b:.0f}x{b:.0f}x{t:.1f}      {r['p_util']:<10.1%}{r['m_util']:<10.1%}{r['combined']:<10.3f}{status}")

final_steel = steel_strength_check(50.0, 3.5, Mu, Vu, Pu)
assert final_steel['combined'] <= 1.0, "最終試設應該要PASS"
print(f"\n[PASS] HSS50x50x3.5cm 通過檢核(交互作用值={final_steel['combined']:.3f})")
print(f"發現: 加大外徑(40->50cm)比單純加厚管壁(1.6->5.0cm)更有效——")
print(f"      40x5.0cm仍FAIL(交互作用{steel_strength_check(40,5.0,Mu,Vu,Pu)['combined']:.3f}),")
print(f"      50x3.5cm卻PASS, 證實深度對抗彎效率的影響比材料量本身更關鍵。")

斷面                P利用率      M利用率      交互作用      結果    
HSS40x40x1.6      3.0%      289.4%    2.908     FAIL
HSS40x40x2.5      1.9%      194.0%    1.950     FAIL
HSS40x40x3.5      1.4%      146.0%    1.468     FAIL
HSS40x40x5.0      1.0%      110.8%    1.113     FAIL
HSS50x50x3.5      1.1%      90.1%     0.906     PASS

[PASS] HSS50x50x3.5cm 通過檢核(交互作用值=0.906)
發現: 加大外徑(40->50cm)比單純加厚管壁(1.6->5.0cm)更有效——
      40x5.0cm仍FAIL(交互作用1.113),
      50x3.5cm卻PASS, 證實深度對抗彎效率的影響比材料量本身更關鍵。


## 第 6 課:兩種材料並列比較——同一組需求力,不同的答案

這就是「可抽換檢核模組」真正的價值:分析端(需求力)完全共用,
只有檢核端跟著材料換,而且換材料後連「該往哪個方向調整斷面」的
工程直覺都不一樣。

In [7]:
print(f"{'材料':<8}{'斷面':<20}{'M利用率':<10}{'governing模式'}")
print(f"{'RC':<8}{rc_result['section']:<20}{rc_result['m_util']:<10.1%}{'彎矩(遠低於容量)'}")
print(f"{'Steel':<8}{final_steel['section']:<20}{final_steel['m_util']:<10.1%}{'組合彎矩+軸力交互作用'}")

print(f"\n需求力(兩種材料完全相同, 因為分析端沒有變):")
print(f"  Mu={Mu:.3f} kN-m, Vu={Vu:.3f} kN, Pu={Pu:.2f} kN")

材料      斷面                  M利用率      governing模式
RC      40x40cm             15.8%     彎矩(遠低於容量)
Steel   HSS50x50x3.5cm      90.1%     組合彎矩+軸力交互作用

需求力(兩種材料完全相同, 因為分析端沒有變):
  Mu=22.608 kN-m, Vu=0.000 kN, Pu=147.60 kN


## 總結表

In [8]:
print("="*55)
print("Case-03.6b 可抽換檢核模組示範總結")
print("="*55)
print(f"{'需求力Mu(兩材料共用)':<26}{Mu:.3f} kN-m")
print(f"{'RC最終斷面':<26}{rc_result['section']}")
print(f"{'RC彎矩利用率':<26}{rc_result['m_util']:.1%}")
print(f"{'鋼結構第一次嘗試':<26}HSS40x40x1.6cm (FAIL)")
print(f"{'鋼結構最終斷面':<26}{final_steel['section']}")
print(f"{'鋼結構組合交互作用值':<26}{final_steel['combined']:.3f}")
print()
print("Case-03.6b [PASS] -- 分析/檢核分離架構驗證成立, 可繼續Case-04")

Case-03.6b 可抽換檢核模組示範總結
需求力Mu(兩材料共用)              22.608 kN-m
RC最終斷面                    40x40cm
RC彎矩利用率                   15.8%
鋼結構第一次嘗試                  HSS40x40x1.6cm (FAIL)
鋼結構最終斷面                   HSS50x50x3.5cm
鋼結構組合交互作用值                0.906

Case-03.6b [PASS] -- 分析/檢核分離架構驗證成立, 可繼續Case-04
